# Annotate VEP variant table with consequence from the VAT
A preprocessing step

In [ ]:
import urllib
from pathlib import Path
from tqdm.notebook import tqdm
import pandas as pd

In [ ]:
bucket = Path.home() / 'workspace/scalable-vus-validation'

input_vep_url = 'https://api.data.igvf.org/tabular-files/IGVFFI1328FVZD/@@download/IGVFFI1328FVZD.csv.gz'
input_vep_file = bucket / 'variant_tables/IGVFFI1328FVZD.csv.gz'
gene_vat_dir = bucket / 'vat'
gene_cohort_dir = bucket / 'gene_cohorts'
output_vep_file = bucket / 'variant_tables/annotated_filtered_VEP_calibrations.tsv.gz'

In [ ]:
# Download VEP table if it does not exist
if not input_vep_file.exists():
    print(f'Downloading {input_vep_url} to {input_vep_file}')
    with urllib.request.urlopen(input_vep_url) as in_f:
        with input_vep_file.open('wb') as out_f:
            out_f.write(in_f.read())

In [ ]:
# Load VEP table
vep_df = pd.read_csv(input_vep_file, dtype=str)


In [ ]:
# Filter to genes with cohorts
cohort_genes = {gene for gene in tqdm(vep_df['gene_symbol'].drop_duplicates()) if Path(gene_cohort_dir, f'{gene} cohort.parquet').exists()}
display(cohort_genes)

In [ ]:
filtered_vep_df = vep_df[
    vep_df['gene_symbol'].isin(cohort_genes)
].assign(
    POS = lambda df: pd.to_numeric(df['POS'], downcast='integer')
)

In [ ]:
# Check VAT table files
for gene in filtered_vep_df['gene_symbol'].drop_duplicates():
    if not Path(gene_vat_dir, f"{gene}_vat.tsv").exists():
        print(f'VAT for {gene} not found')


In [ ]:
annotated_dfs = {}
for gene, gene_df in tqdm(filtered_vep_df.groupby('gene_symbol')):
    gene_vat = pd.read_table(
        Path(gene_vat_dir, f"{gene}_vat.tsv"),
        dtype=str
    ).assign(pos = lambda df: pd.to_numeric(df['position'], downcast='integer'))
    
    annotated_dfs[gene] = gene_df.merge(
        gene_vat,
        how = 'left',
        left_on = ['#CHROM', 'POS', 'REF', 'ALT'],
        right_on = ['contig', 'pos', 'ref_allele', 'alt_allele'],
        suffixes = (None, '_VAT')
    )

In [ ]:
annotated_df = pd.concat(annotated_dfs, names=['Gene'])

In [ ]:
annotated_df[['gene_symbol', 'consequence']].value_counts(dropna=False).sort_index()

In [ ]:
annotated_df.to_csv(output_vep_file, sep='\t')